In [25]:
import pandas as pd
df = pd.read_csv('C:/Users/USER/Desktop/sales-analytics-dashboard/data/raw/superstore.csv.csv',encoding = 'latin-1')
print(f'Starting Shape:{df.shape}')
df.head()

Starting Shape:(9800, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## Data Cleaning

### Issues identified in EDA:
- Order Date and Ship Date are strings — need to convert to datetime
- Postal Code is float64 with 11 nulls — need to convert to string
- Column names need standardizing to lowercase with underscores
- Need to engineer new columns: profit_margin, days_to_ship, order_month, order_year

In [26]:
#fixing the date columns
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')

print(df[['Order Date', 'Ship Date']].dtypes)
print(df[['Order Date', 'Ship Date']].head())

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object
  Order Date  Ship Date
0 2017-11-08 2017-11-11
1 2017-11-08 2017-11-11
2 2017-06-12 2017-06-16
3 2016-10-11 2016-10-18
4 2016-10-11 2016-10-18


In [27]:
#fixing the postal code
df['Postal Code'] = df['Postal Code'].fillna(0).astype(int).astype(str)
df['Postal Code'] = df['Postal Code'].replace('0', 'Unknown')

print(df['Postal Code'].head(10))

0    42420
1    42420
2    90036
3    33311
4    33311
5    90032
6    90032
7    90032
8    90032
9    90032
Name: Postal Code, dtype: str


In [28]:
#Standardize column names 
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(df.columns.tolist())

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub-category', 'product_name', 'sales']


In [29]:
#changing the column sub-category to sub_category
df.columns = df.columns.str.lower().str.replace('-','_')

In [30]:
#Engineer new columns
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)

print(df[['order_date', 'ship_date', 'days_to_ship', 'order_year', 'order_month']].head())

  order_date  ship_date  days_to_ship  order_year order_month
0 2017-11-08 2017-11-11             3        2017     2017-11
1 2017-11-08 2017-11-11             3        2017     2017-11
2 2017-06-12 2017-06-16             4        2017     2017-06
3 2016-10-11 2016-10-18             7        2016     2016-10
4 2016-10-11 2016-10-18             7        2016     2016-10


In [16]:
#removing the duplicates
before = df.shape[0]
df = df.drop_duplicates(subset='order_id')
after = df.shape[0]

print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Duplicates removed: {before - after}")

Rows before: 9800
Rows after: 4922
Duplicates removed: 4878


we need to pause here because this result is actually a problem. Removing 4,878 rows is almost half my dataset and that's a red flag worth investigating before we continue.

The Superstore dataset has one row per product per order — meaning if a customer ordered 3 different products in one order, that order ID appears 3 times with different products and sales amounts. Those are not duplicates, they are legitimate separate line items.
By dropping duplicates on order_id alone I accidentally deleted real data. I'll need to reload the raw data again so i won't work with the already modified one.


In [ ]:
# Load the original raw file fresh 
df_raw = pd.read_csv('C:/Users/USER/Desktop/sales-analytics-dashboard/data/raw/superstore.csv.csv', encoding='latin-1')

# Standardize column names so we can reference them easily
df_raw.columns = df_raw.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')

# Check how many times each order_id appears
order_counts = df_raw['order_id'].value_counts()
print(order_counts.head(10))
print(f"\nMax products in one order: {order_counts.max()}")
print(f"Average products per order: {order_counts.mean():.1f}")

order_id
CA-2018-100111    14
CA-2018-157987    12
US-2017-108504    11
CA-2017-165330    11
CA-2016-131338    10
US-2016-126977    10
CA-2017-105732    10
CA-2018-117457     9
CA-2015-106439     9
CA-2016-164882     9
Name: count, dtype: int64

Max products in one order: 14
Average products per order: 2.0


What this output tells us

Order CA-2018-100111 appears 14 times — meaning one customer ordered 14 different products in a single order
The average order has 2 products in it
These are all legitimate rows, not duplicates — each row is a different product in the same order

So when we did drop_duplicates(subset='order_id') we kept only the first product from each order and deleted all the others. That's why we lost 4,878 rows — we accidentally deleted real sales data.


In [20]:
# Checking  for rows where everything is identical
true_duplicates = df_raw.duplicated()
print(f"True duplicates: {true_duplicates.sum()}")

True duplicates: 0


In [ ]:
# Checking if any order has the exact same product twice
order_product_dupes = df_raw.duplicated(subset=['order_id', 'product_id'])
print(f"Same product appearing twice in same order: {order_product_dupes.sum()}")

Same product appearing twice in same order: 8


The fix of cell 16


In [ ]:
before = df.shape[0]

# Dropping only rows where the same product appears twice in the same order
# These are true data entry errors, not legitimate line items
df = df.drop_duplicates(subset=['order_id', 'product_id'])

after = df.shape[0]

print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"True duplicates removed: {before - after}")

Rows before: 9800
Rows after: 9792
True duplicates removed: 8


Saving the cleaned data

In [32]:
df.to_csv('../data/processed/sales_clean.csv', index=False)
print(f"Cleaned dataset saved: {df.shape[0]} rows, {df.shape[1]} columns")

Cleaned dataset saved: 9792 rows, 21 columns
